In [2]:
!pip -q install transformers peft bitsandbytes accelerate chromadb sentence-transformers gradio beautifulsoup4
from google.colab import userdata, drive
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os, sys, json
REPO = "/content/quizgen-llama3"
if os.path.exists(REPO):
    !git -C {REPO} pull
else:
    !git clone https://github.com/Somtochukwu438/quizgen-llama3.git {REPO}
sys.path.append(REPO)
from src.generate import load_model, generate_quiz
from src.notes import load_notes
from src.rag import NotesIndex
OUT = "/content/drive/MyDrive/quizgen"

Cloning into '/content/quizgen-llama3'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 71 (delta 8), reused 15 (delta 2), pack-reused 45 (from 1)
Receiving objects: 100% (71/71), 42.48 MiB | 20.90 MiB/s, done.
Resolving deltas: 100% (25/25), done.


In [4]:
from datasets import load_dataset

model, tok = load_model(f"{REPO}/adapters")

test = load_dataset("allenai/sciq")["test"].filter(lambda x: x["support"].strip() != "")
eval_set = test.shuffle(seed=42).select(range(300))
with open(f"{OUT}/finetuned_outputs.jsonl") as f:
    saved = [json.loads(line) for line in f][:8]

recs = generate_quiz(model, tok, eval_set["support"][:8], max_tries=1)
matches = sum(r["raw"] == s["raw"] for r, s in zip(recs, saved))
print(f"{matches}/8 outputs identical to the saved evaluation run")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (12 outputs left out)

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


8/8 outputs identical to the saved evaluation run


In [5]:
passages = load_notes(f"{REPO}/data/notes")
index = NotesIndex(passages)
print(len(passages), "passages indexed")
for topic in ["sleep stages", "how we feel pain", "brain imaging methods", "sense of smell"]:
    hits = index.search(topic, k=2)
    print(f"{topic!r} -> " + " | ".join(h["section"] for h in hits))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

12 passages indexed
'sleep stages' -> Sleep architecture | Functions and disorders
'how we feel pain' -> Body and pain | Organizing the nervous system
'brain imaging methods' -> Methods | Major brain systems
'sense of smell' -> Taste and smell | Body and pain


In [6]:
hits = index.search("sleep stages", k=3)
quiz = generate_quiz(model, tok, [h["text"] for h in hits])
for h, q in zip(hits, quiz):
    print(f"[{h['week'][:7]} · {h['section']}]  attempts: {q['attempts']}")
    if q["parsed"]:
        p = q["parsed"]
        print("Q:", p["question"], "| A:", p["answer"], "| D:", p["distractors"])
    else:
        print("FAILED:", q["error"])
    print("-" * 70)

[Week 9: · Sleep architecture]  attempts: 2
Q: Which type of sleep includes cortical activation, vivid dreaming, rapid eye movements, and muscle atonia? | A: rem sleep | D: ['nrem sleep', 'sleep', 'parasitic sleep']
----------------------------------------------------------------------
[Week 9: · Functions and disorders]  attempts: 1
Q: What supports learning, emotional regulation, metabolic health, immune function, and safety? | A: sleep | D: ['exercise', 'nutrition', 'medication']
----------------------------------------------------------------------
[Week 9: · Two-process regulation]  attempts: 1
Q: What is the name of the nucleus that coordinates daily rhythms using light input? | A: suprachiasmatic nucleus | D: ['hypothalamus', 'hypothalamus nucleus', 'hypothalamus gland']
----------------------------------------------------------------------
